<a href="https://colab.research.google.com/github/Jirtus-sanasam/MLP-Diabetes/blob/main/DiabetesMLP_rawdata.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
df = pd.read_csv('/content/diabetes_data2.csv')

In [ ]:
# Replacing 0 as null value
cols = ['Glucose','BloodPressure','SkinThickness','Insulin','BMI']
df[cols] = df[cols].replace(0, np.nan)

In [ ]:
# Imputing Null Value with KNNImputer
from sklearn.impute import KNNImputer
knn_imputer = KNNImputer(n_neighbors=5)
df[cols] = knn_imputer.fit_transform(df[cols])

In [ ]:
# Cap outliers using IQR
def cap_outliers(df, col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df[col] = df[col].clip(lower, upper)
    return df

for col in df.columns[:-1]:  # exclude target
    df = cap_outliers(df, col)

In [ ]:

# FEATURE ENGINEERING

# --- 1. Interaction Features ---
df['Glucose_BMI'] = df['Glucose'] * df['BMI']
df['Glucose_Insulin'] = df['Glucose'] * df['Insulin']
df['BMI_Age'] = df['BMI'] * df['Age']
df['Insulin_SkinThickness'] = df['Insulin'] * df['SkinThickness']

# --- 2. Ratio Features (avoid division by zero) ---
df['Glucose_Insulin_Ratio'] = df['Glucose'] / (df['Insulin'] + 1e-6)
df['BMI_Age_Ratio'] = df['BMI'] / (df['Age'] + 1e-6)

# --- 3. Polynomial Features ---
df['Glucose_sq'] = df['Glucose'] ** 2
df['BMI_sq'] = df['BMI'] ** 2
df['Age_sq'] = df['Age'] ** 2

# --- 4. Age Binning ---
df['Age_Group'] = pd.cut(
    df['Age'],
    bins=[0, 30, 45, 60, 100],
    labels=['Young', 'Middle', 'Senior', 'Elderly']
)
# One-hot encoding with integer output
df = pd.get_dummies(df, columns=['Age_Group'], drop_first=True, dtype=int)

# --- 5. BMI Category ---
df['BMI_Category'] = pd.cut(
    df['BMI'],
    bins=[0, 18.5, 24.9, 29.9, 100],
    labels=['Underweight', 'Normal', 'Overweight', 'Obese']
)

df = pd.get_dummies(df, columns=['BMI_Category'], drop_first=True, dtype=int)

# --- 6. Glucose Category ---
df['Glucose_Level'] = pd.cut(
    df['Glucose'],
    bins=[0, 99, 125, 300],
    labels=['Normal', 'Prediabetes', 'Diabetes']
)

df = pd.get_dummies(df, columns=['Glucose_Level'], drop_first=True, dtype=int)

# --- 7. Insulin Resistance (HOMA-IR) ---
df['HOMA_IR'] = (df['Glucose'] * df['Insulin']) / 405
# --- 8. Pregnancy Risk Flag ---
df['High_Pregnancies'] = (df['Pregnancies'] > 3).astype(int)

# --- 9. Log Transform (handle skewness) ---
df['Log_Insulin'] = np.log1p(df['Insulin'])
df['Log_DiabetesPedigreeFunction'] = np.log1p(df['DiabetesPedigreeFunction'])

# --- 10. Composite Risk Score ---
df['Risk_Score'] = (
    (df['Glucose'] / 100) +
    (df['BMI'] / 25) +
    (df['Age'] / 50) +
    (df['DiabetesPedigreeFunction'])
)

# --- 11. Final Safety Check (convert any leftover bool → int) ---
bool_cols = df.select_dtypes(include='bool').columns
if not bool_cols.empty:
    for col in bool_cols.unique(): # Iterate over unique column names
        # Ensure we are only trying to convert actual boolean columns that might still exist
        if df[col].dtype == 'bool':
            df[col] = df[col].astype(int)
# --- 12. Verify Data Types ---
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 31 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Pregnancies                   768 non-null    float64
 1   Glucose                       768 non-null    float64
 2   BloodPressure                 768 non-null    float64
 3   SkinThickness                 768 non-null    float64
 4   Insulin                       768 non-null    float64
 5   BMI                           768 non-null    float64
 6   DiabetesPedigreeFunction      768 non-null    float64
 7   Age                           768 non-null    float64
 8   Outcome                       768 non-null    int64  
 9   Glucose_BMI                   768 non-null    float64
 10  Glucose_Insulin               768 non-null    float64
 11  BMI_Age                       768 non-null    float64
 12  Insulin_SkinThickness         768 non-null    float64
 13  Gluco

In [7]:
X = df.drop('Outcome', axis=1)
Y = df['Outcome']

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42, stratify=Y
)

In [9]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

In [10]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train_scaled, y_train)

print("Before SMOTE:", dict(pd.Series(y_train).value_counts()))
print("After SMOTE: ", dict(pd.Series(y_train_sm).value_counts()))

Before SMOTE: {0: np.int64(400), 1: np.int64(214)}
After SMOTE:  {0: np.int64(400), 1: np.int64(400)}


In [11]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

def build_mlp(input_dim):
    model = Sequential([
        Dense(128, activation='relu', input_shape=(input_dim,)),
        BatchNormalization(),
        Dropout(0.3),

        Dense(64, activation='relu'),
        BatchNormalization(),
        Dropout(0.3),

        Dense(32, activation='relu'),
        Dropout(0.2),

        Dense(1, activation='sigmoid')  # Binary classification
    ])
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

In [12]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score
import numpy as np

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_accuracy, cv_auc, cv_f1 = [], [], []

X_full = scaler.fit_transform(X)   # scale full X for CV
Y_arr  = Y.values

for fold, (train_idx, val_idx) in enumerate(skf.split(X_full, Y_arr)):
    print(f"\n--- Fold {fold+1} ---")

    X_tr, X_val = X_full[train_idx], X_full[val_idx]
    y_tr, y_val = Y_arr[train_idx],  Y_arr[val_idx]

    # Apply SMOTE per fold
    smote = SMOTE(random_state=42)
    X_tr_sm, y_tr_sm = smote.fit_resample(X_tr, y_tr)

    # Build fresh model each fold
    model = build_mlp(X_tr_sm.shape[1])

    early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

    model.fit(
        X_tr_sm, y_tr_sm,
        epochs=100,
        batch_size=32,
        validation_data=(X_val, y_val),
        callbacks=[early_stop],
        verbose=0
    )

    y_pred_prob = model.predict(X_val).flatten()
    y_pred      = (y_pred_prob >= 0.5).astype(int)

    acc = accuracy_score(y_val, y_pred)
    auc = roc_auc_score(y_val, y_pred_prob)
    f1  = f1_score(y_val, y_pred)

    cv_accuracy.append(acc)
    cv_auc.append(auc)
    cv_f1.append(f1)

    print(f"  Accuracy: {acc:.4f} | AUC: {auc:.4f} | F1: {f1:.4f}")

print("\n========== CV Summary ==========")
print(f"Accuracy : {np.mean(cv_accuracy):.4f} ± {np.std(cv_accuracy):.4f}")
print(f"AUC-ROC  : {np.mean(cv_auc):.4f} ± {np.std(cv_auc):.4f}")
print(f"F1 Score : {np.mean(cv_f1):.4f} ± {np.std(cv_f1):.4f}")


--- Fold 1 ---


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
  Accuracy: 0.7922 | AUC: 0.8511 | F1: 0.7333

--- Fold 2 ---


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
  Accuracy: 0.7922 | AUC: 0.8672 | F1: 0.7419

--- Fold 3 ---


...

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


  Accuracy: 0.7338 | AUC: 0.8476 | F1: 0.6772

--- Fold 4 ---
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
  Accuracy: 0.7386 | AUC: 0.8340 | F1: 0.7015

--- Fold 5 ---


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
  Accuracy: 0.7059 | AUC: 0.8036 | F1: 0.6565

========== CV Summary ==========
Accuracy : 0.7525 ± 0.0343
AUC-ROC  : 0.8407 ± 0.0214
F1 Score : 0.7021 ± 0.0324


In [13]:
# Train final model on SMOTE-augmented train data
final_model = build_mlp(X_train_sm.shape[1])

early_stop = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)

history = final_model.fit(
    X_train_sm, y_train_sm,
    epochs=150,
    batch_size=32,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)

# Evaluate on held-out test set
y_test_prob = final_model.predict(X_test_scaled).flatten()
y_test_pred = (y_test_prob >= 0.5).astype(int)

from sklearn.metrics import classification_report, confusion_matrix

print("\n=== Final Test Set Results ===")
print("Accuracy :", accuracy_score(y_test, y_test_pred))
print("AUC-ROC  :", roc_auc_score(y_test, y_test_prob))
print("\n", classification_report(y_test, y_test_pred))

Epoch 1/150


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


23/23 ━━━━━━━━━━━━━━━━━━━━ 12s 95ms/step - accuracy: 0.6583 - loss: 0.6425 - val_accuracy: 0.6875 - val_loss: 0.5475
Epoch 2/150
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.7278 - loss: 0.5545 - val_accuracy: 0.7625 - val_loss: 0.5209
Epoch 3/150
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.7444 - loss: 0.4856 - val_accuracy: 0.8375 - val_loss: 0.4852
Epoch 4/150
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.7472 - loss: 0.4955 - val_accuracy: 0.8375 - val_loss: 0.4556
Epoch 5/150
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 0.7778 - loss: 0.4818 - val_accuracy: 0.8750 - val_loss: 0.4295
Epoch 6/150
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.7625 - loss: 0.4580 - val_accuracy: 0.9000 - val_loss: 0.4026
Epoch 7/150
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.7736 - loss: 0.4668 - val_accuracy: 0.9125 - val_loss: 0.3895
Epoch 8/150
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 0.7528 - loss: 0.4594 - val_accuracy: 0.9125 - val_